In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.cluster import KMeans
from sklearn.compose import ColumnTransformer
import matplotlib.pyplot as plt
from datetime import datetime
from sklearn.cluster import DBSCAN
from sklearn.cluster import OPTICS
from sklearn.mixture import GaussianMixture
from sklearn.cluster import AgglomerativeClustering
import scipy.cluster.hierarchy as sch
from sklearn.neighbors import NearestNeighbors
from sklearn.decomposition import PCA
import seaborn as sns
import plotly.graph_objects as go
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score

In [ ]:
from pathlib import Path

# CRITICAL: .resolve() gives an absolute path — Path('.').parent == Path('.') without it (doesn't go up directories!)
_HERE = Path(".").resolve()

# ── customer features (output from cleaning_data.ipynb) ─────────────────────
_candidates_custom = [
    _HERE / "customer_features_final_clean.csv",
    _HERE.parent / "customer_features_final_clean.csv",
    _HERE.parent.parent / "customer_features_final_clean.csv",
    _HERE / "customer_features_with_clusters.csv",
    _HERE.parent / "customer_features_with_clusters.csv",
    _HERE.parent.parent / "customer_features_with_clusters.csv",  # data science in action/
    _HERE / "io" / "customer_features_with_clusters.csv",
    _HERE.parent / "io" / "customer_features_with_clusters.csv",
]

custom_path = next((p for p in _candidates_custom if p.exists()), None)
if custom_path is None:
    raise FileNotFoundError(
        "Customer features file not found. "
        "Run src/cleaning_data.ipynb first, or copy "
        "customer_features_with_clusters.csv to the project root."
    )

print(f"Loading customer features from: {custom_path}")
custom = pd.read_csv(custom_path)
print(f"Shape: {custom.shape}")

# ── master transactions (optional, for additional features) ─────────────────
master = None
for _candidate in [
    _HERE / "master_transactions.csv",
    _HERE.parent / "master_transactions.csv",
    _HERE.parent.parent / "master_transactions.csv",        # data science in action/
    _HERE.parent.parent.parent / "master_transactions.csv", # Desktop/ (fallback)
    _HERE / "io" / "master_transactions.csv",
    _HERE.parent / "io" / "master_transactions.csv",
]:
    if _candidate.exists():
        master = pd.read_csv(_candidate)
        print(f"Master transactions loaded from: {_candidate}  shape: {master.shape}")
        break

if master is None:
    print("master_transactions.csv not found — proceeding with customer features only.")

# Feature engineering and preprocessing

In [ ]:
if master is not None and {'list_price', 'Quantity', 'customer_id'}.issubset(master.columns):
    # 1. Calculate the actual Transaction Value for every row
    master['transaction_value'] = master['list_price'] * master['Quantity']

    # 2. Group by Customer to find the 'Net Spenders'
    customer_financials = master.groupby('customer_id').agg(
        total_net_spend=('transaction_value', 'sum'),
        total_items_handled=('Quantity', 'sum'),
        transaction_count=('customer_id', 'count')
    )

    # 3. Identify customers where Total Net Spend is negative
    negative_spend_ids = customer_financials[customer_financials['total_net_spend'] < 0].index.tolist()

    # 4. Extract their full history from the Master DF
    negative_audit_df = master[master['customer_id'].isin(negative_spend_ids)].copy()
    if 'Date' in negative_audit_df.columns:
        negative_audit_df = negative_audit_df.sort_values(by=['customer_id', 'Date'])
    else:
        negative_audit_df = negative_audit_df.sort_values(by=['customer_id'])

    # 5. Export to CSV
    negative_audit_df.to_csv('negative_spend_audit_detailed.csv', index=False)

    print(f"Found {len(negative_spend_ids)} customers with a negative total balance.")
    print("Detailed transaction history saved to 'negative_spend_audit_detailed.csv'")

    # Quick look at the 'Top Returners' by value
    print("\nCustomers with the most negative balances:")
    print(customer_financials.loc[negative_spend_ids].sort_values('total_net_spend').head())
else:
    print('Skipping negative-spend audit: transactional columns not available.')

### 52 customers received multiple (2-3) returns for each product they bought and returned. The subsequent returns occured usually a week or two after the initial return

In [ ]:
# Remove columns that could conflict with recalculated fields in this notebook
custom = custom.drop(columns=['cluster', 'recency_days', 'Gender', 'province'], errors='ignore')

In [ ]:
if master is not None and {'list_price', 'Quantity', 'customer_id'}.issubset(master.columns):
    master['transaction_value'] = master['list_price'] * master['Quantity']

    # Calculate total lifetime value (LTV) for every customer in the dataset
    customer_check = master.groupby('customer_id')['transaction_value'].sum()

    # Keep only IDs with non-negative total spend
    valid_customer_ids = customer_check[customer_check >= 0].index

    # Create a clean master dataframe excluding negative spenders
    master = master[master['customer_id'].isin(valid_customer_ids)].copy()
else:
    print('Skipping master-level filtering: transactional dataset not available.')

In [ ]:
# Convert Date only when a transactional dataset is available
if master is not None and 'Date' in master.columns:
    master['Date'] = pd.to_datetime(master['Date'], format='%d%b%Y', errors='coerce')
else:
    print('Skipping Date conversion: no transactional Date column found.')

In [ ]:
# Build recency and demographics from transactional data when available
if master is not None and {'customer_id', 'Date'}.issubset(master.columns):
    # Reference date for Recency
    ref_date = master['Date'].max()

    # Recency: days since latest transaction
    recency_df = master.groupby('customer_id')['Date'].max().reset_index()
    recency_df['recency_days'] = (ref_date - recency_df['Date']).dt.days

    # Optional demographics from transactions
    available_demo_cols = [c for c in ['customer_id', 'Gender', 'province'] if c in master.columns]
    demographics = master[available_demo_cols].drop_duplicates('customer_id') if len(available_demo_cols) > 1 else None
else:
    recency_df = None
    demographics = None
    print('Transactional recency/demographics unavailable; using customer dataset fields only.')

In [ ]:
# BUILD FINAL TABLE FOR CLUSTERING
df_final = custom.copy()

# Merge recency from transactions when available
if recency_df is not None and {'customer_id', 'recency_days'}.issubset(recency_df.columns):
    df_final = df_final.merge(recency_df[['customer_id', 'recency_days']], on='customer_id', how='left', suffixes=('', '_from_tx'))
    if 'recency_days_from_tx' in df_final.columns:
        df_final['recency_days'] = df_final['recency_days_from_tx'].combine_first(df_final.get('recency_days'))
        df_final = df_final.drop(columns=['recency_days_from_tx'])

# Merge demographics when available
if demographics is not None and 'customer_id' in demographics.columns:
    demo_cols = [c for c in demographics.columns if c != 'customer_id']
    if demo_cols:
        df_final = df_final.merge(demographics, on='customer_id', how='left', suffixes=('', '_from_tx'))
        for c in demo_cols:
            tx_col = f'{c}_from_tx'
            if tx_col in df_final.columns:
                df_final[c] = df_final[tx_col].combine_first(df_final.get(c))
                df_final = df_final.drop(columns=[tx_col])

In [ ]:
# Handle missing recency
if 'recency_days' in df_final.columns:
    # If a customer has no recency, set a high value
    df_final['recency_days'] = df_final['recency_days'].fillna(600)
else:
    # If recency is completely unavailable, create a default value
    df_final['recency_days'] = 600
    print('recency_days not found; created default recency_days = 600 for all rows.')

In [ ]:
# PREPROCESSING
# Use an expanded feature set from the new engineered dataset
candidate_num_cols = [
    'age_years', 'net_amount', 'transaction_count', 'return_rate_tx',
    'nl_open_rate', 'nl_click_rate', 'recency_days',
    'gross_spend', 'return_value', 'avg_ticket',
    'discounted_tx_rate', 'purchase_frequency_per_month', 'ecommerce_share'
]
num_cols = [c for c in candidate_num_cols if c in df_final.columns]

candidate_cat_cols = ['Gender', 'dominant_channel']
cat_cols = [c for c in candidate_cat_cols if c in df_final.columns]

if not num_cols:
    raise ValueError('No numeric columns available for clustering after merge.')

transformers = [('num', StandardScaler(), num_cols)]
if cat_cols:
    transformers.append(
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols)
    )

preprocessor = ColumnTransformer(transformers=transformers)

# Drop rows missing selected modeling features
df_final = df_final.dropna(subset=num_cols + cat_cols)

X_scaled = preprocessor.fit_transform(df_final)

print('Numeric features used:', num_cols)
print('Categorical features used:', cat_cols)
print('Rows available for clustering:', len(df_final))

In [ ]:
# CHECK DATA ANOMALIES
print("=" * 100)
print("DATA QUALITY CHECK")
print("=" * 100)

# 1. Missing values
print("\n1. MISSING VALUES (NULL):")
null_counts = df_final[num_cols].isnull().sum()
if null_counts.sum() == 0:
    print("   ✓ No missing values in numeric features used for clustering")
else:
    print("   ⚠ Missing values detected:")
    print(null_counts[null_counts > 0])

# 2. Unexpected negative values
print("\n2. NEGATIVE VALUES:")
for col in ['gross_spend', 'net_amount', 'avg_ticket']:
    if col in df_final.columns:
        neg_count = (df_final[col] < 0).sum()
        if neg_count > 0:
            print(f"   ⚠ {col}: {neg_count} customers with negative values")
        else:
            print(f"   ✓ {col}: No negative values")

# 3. Logical inconsistencies (gross_spend < return_value is impossible!)
print("\n3. LOGICAL INCONSISTENCIES:")
incoherent = (df_final['gross_spend'] < df_final['return_value']).sum()
if incoherent > 0:
    print(f"   ⚠ {incoherent} customers with return_value > gross_spend (IMPOSSIBLE!)")
    print("   Details:")
    print(df_final[df_final['gross_spend'] < df_final['return_value']][['customer_id', 'gross_spend', 'return_value', 'net_amount']])
else:
    print("   ✓ No logical inconsistency (gross_spend >= return_value in all cases)")

# 4. Margin distribution to verify outliers
print("\n4. NET MARGIN DISTRIBUTION BY CLUSTER:")
for cluster_id in sorted(df_final['cluster'].unique()):
    cluster_data = df_final[df_final['cluster'] == cluster_id].copy()
    cluster_data['margin_pct'] = (
        (cluster_data['gross_spend'] - cluster_data['return_value']) 
        / cluster_data['gross_spend'] * 100
    )
    
    margin_stats = cluster_data['margin_pct'].describe()
    
    print(f"\n   Cluster {cluster_id} ({len(cluster_data)} customers):")
    print(f"      Mean: {margin_stats['mean']:.2f}%")
    print(f"      Min: {margin_stats['min']:.2f}%")
    print(f"      Max: {margin_stats['max']:.2f}%")
    print(f"      Std Dev: {margin_stats['std']:.2f}%")
    
    # Count outliers (values outside mean ± 3*std)
    outlier_count = (
        (cluster_data['margin_pct'] < margin_stats['mean'] - 3*margin_stats['std']) |
        (cluster_data['margin_pct'] > margin_stats['mean'] + 3*margin_stats['std'])
    ).sum()
    if outlier_count > 0:
        print(f"      ⚠ Outliers detected (mean±3σ): {outlier_count}")
    else:
        print(f"      ✓ No statistical outliers")

print("\n" + "=" * 100)
print("CONCLUSION: Data checked and validated!")
print("=" * 100)

In [ ]:
# ANOMALY INVESTIGATION (return_value > gross_spend)
anomalies = df_final[df_final['gross_spend'] < df_final['return_value']][['customer_id', 'cluster', 'gross_spend', 'return_value', 'net_amount', 'transaction_count']]
print(f"\nANOMALIES DETECTED: {len(anomalies)} customers")
print(anomalies.to_string())

# Explanation: these customers had returns greater than purchases in value
# They may be credits, refunds, or special service adjustments
print("\nNote: These customers have return_value > gross_spend,")
print("which means they received more credits/returns than they originally spent.")
print("They could be transfer refunds or special credits. We exclude them from margin calculation.")

# Recalculate margin excluding anomalies
print("\n" + "=" * 100)
print("NET MARGIN BY CLUSTER (Excluding 9 anomalies)")
print("=" * 100)

df_clean = df_final[(df_final['gross_spend'] >= df_final['return_value'])].copy()
print(f"Valid customers: {len(df_clean)} (excluding 9 anomalies)")

margin_clean = df_clean.groupby('cluster').agg(
    customer_count=('customer_id', 'count'),
    total_gross_spend=('gross_spend', 'sum'),
    total_return_value=('return_value', 'sum'),
).reset_index()

margin_clean['net_margin_pct'] = (
    (margin_clean['total_gross_spend'] - margin_clean['total_return_value']) 
    / margin_clean['total_gross_spend'] * 100
).round(2)

print("\n")
for _, row in margin_clean.sort_values('net_margin_pct', ascending=False).iterrows():
    print(f"Cluster {int(row['cluster'])}: Net Margin {row['net_margin_pct']:6.2f}% | {int(row['customer_count']):,} customers | Spend: EUR {row['total_gross_spend']:,.0f}")

In [ ]:
validated_margin = margin_analysis.sort_values('net_margin_pct', ascending=False).copy()
anomaly_count = len(anomalies)

# Cluster profiles derived from K-Means feature summary (authoritative behavioral source):
#   C0: discount_rate=74.6%, recency=207 days, return_value≈0, ecommerce=0.8%  → Deal Hunters
#   C1: recency=255 days (most inactive), discount=6.5%, gross_spend=€328       → Dormant Full-Price Shoppers
#   C2: ecommerce=95.1%, high return_value, avg_ticket=€49, freq=1.93/month    → E-Commerce Serial Returners
#   C3: recency=113 days (most active), gross_spend=€1,191 (highest), ecomm=0% → Active In-Store Champions
#
# Note: margin analysis alone (which only captures return rate) would not distinguish C1 from C3,
# since both show ~98.35% net margin. Their behavioral divergence is visible only in the full
# feature space: recency (255 vs 113 days) and spend level (€328 vs €1,191).
cluster_profiles = {
    0: {
        'name': 'DEAL HUNTERS',
        'description': 'Promotion-driven in-store buyers — 74.6% of transactions on discount, virtually no returns'
    },
    1: {
        'name': 'DORMANT FULL-PRICE SHOPPERS',
        'description': 'Previously loyal full-price buyers now going quiet — 255-day avg recency, lowest discount usage (6.5%)'
    },
    2: {
        'name': 'E-COMMERCE SERIAL RETURNERS',
        'description': 'Online-first buyers with high return volume — 95.1% e-commerce share, low avg ticket (€49)'
    },
    3: {
        'name': 'ACTIVE IN-STORE CHAMPIONS',
        'description': 'Most recently active and highest-spend segment — 113-day recency, €1,191 avg gross spend, 0% e-commerce'
    }
}

print("\n" + "=" * 100)
print("FINAL CLUSTERING REPORT - NET MARGIN ANALYSIS BY CLUSTER")
print("=" * 100)

print("\nVALIDATED DATA:")
print(f"   - Dataset: {len(df_clean)} customers ({anomaly_count} anomalies excluded)")
print(f"   - Total spend: EUR {df_clean['gross_spend'].sum():,.0f}")
print(f"   - Total returns: EUR {df_clean['return_value'].sum():,.0f}")
print(
    f"   - Global average margin: "
    f"{(df_clean['gross_spend'].sum() - df_clean['return_value'].sum()) / df_clean['gross_spend'].sum() * 100:.2f}%"
)

print("\nFULL VALIDATED MARGIN TABLE:")
print(
    validated_margin[
        ['cluster', 'customer_count', 'total_gross_spend', 'total_return_value', 'net_margin_pct']
    ].to_string(index=False)
)

if anomaly_count > 0:
    print("\nFULL LIST OF ANOMALIES EXCLUDED FROM THE CALCULATION:")
    print(anomalies.sort_values(['cluster', 'customer_id']).to_string(index=False))

print("\nK-MEANS SEGMENTATION (4 Clusters):\n")

for _, row in validated_margin.iterrows():
    cid = int(row['cluster'])
    cluster_profile = cluster_profiles.get(
        cid,
        {'name': f'CLUSTER {cid}', 'description': 'Unclassified profile'}
    )
    margin = row['net_margin_pct']
    customers = int(row['customer_count'])
    total_gross_spend = row['total_gross_spend']
    total_return_value = row['total_return_value']
    return_rate = (total_return_value / total_gross_spend * 100) if total_gross_spend else 0
    avg_spend = total_gross_spend / customers if customers else 0

    if margin >= 95:
        profitability = 'EXCELLENT'
        action = 'MAINTAIN AND INCENTIVIZE'
    elif margin >= 90:
        profitability = 'GOOD'
        action = 'MONITOR'
    elif margin >= 50:
        profitability = 'LOW'
        action = 'OPTIMIZE'
    else:
        profitability = 'CRITICAL'
        action = 'URGENT INTERVENTION'

    print(f"   CLUSTER {cid}: {cluster_profile['name']}")
    print(f"   - Description: {cluster_profile['description']}")
    print(f"   - Customers: {customers:,} ({customers / len(df_clean) * 100:.1f}% of total)")
    print(f"   - Net Margin: {margin:.2f}% [{profitability}]")
    print(f"   - Aggregate spend: EUR {total_gross_spend:,.0f}")
    print(f"   - Aggregate returns: EUR {total_return_value:,.0f}")
    print(f"   - Return rate: {return_rate:.2f}%")
    print(f"   - Average customer spend: EUR {avg_spend:.2f}")
    print(f"   - Action: {action}")
    print()

print("STRATEGIC RECOMMENDATIONS:\n")
print("   1. CLUSTER 3 — ACTIVE IN-STORE CHAMPIONS: Protect and grow the highest-value segment")
print("      - Priority VIP programs, exclusive early access, personal styling services")
print("      - Goal: deepen loyalty and raise purchase frequency above 2.3/month")
print()
print("   2. CLUSTER 1 — DORMANT FULL-PRICE SHOPPERS: Reactivation campaigns")
print("      - Personalized win-back sequences (email / DM); highlight new arrivals at full price")
print("      - Avoid discount-led reactivation — this segment historically buys at full price")
print()
print("   3. CLUSTER 0 — DEAL HUNTERS: Margin protection")
print("      - Limit deep-discount exposure; introduce loyalty points as discount substitute")
print("      - Test price-sensitivity: measure elasticity before cutting promotions")
print()
print("   4. CLUSTER 2 — E-COMMERCE SERIAL RETURNERS: Reduce return rate")
print("      - Deploy fit-recommendation tools and detailed size guides on PDPs")
print("      - Introduce restocking fees or minimum keep-rate thresholds for serial returners")
if 2 in validated_margin['cluster'].values:
    current_returns = validated_margin.loc[validated_margin['cluster'] == 2, 'total_return_value'].values
    if len(current_returns):
        print(f"      - Current e-commerce return cost: EUR {current_returns[0]:,.0f}")
print()
print("=" * 100)
print("STATUS: DATA VALIDATED | MARGINS CALCULATED | ANOMALIES IDENTIFIED")
print("=" * 100)

# K-means

In [ ]:
# ELBOW METHOD
inertia = []
K_range = range(1, 11)
for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    inertia.append(km.inertia_)

plt.figure(figsize=(8, 4))
plt.plot(K_range, inertia, 'bx-')
plt.xlabel('Number of Clusters (k)')
plt.ylabel('Inertia')
plt.title('The Elbow Method showing the optimal k')
plt.show()

In [ ]:
# RUNNING K-MEANS 
k_optimal = 4
kmeans = KMeans(n_clusters=k_optimal, random_state=42, n_init=10)
df_final['cluster'] = kmeans.fit_predict(X_scaled)

In [ ]:
# ANALYZE THE RESULTS
cluster_summary = df_final.groupby('cluster')[num_cols].mean()
print(cluster_summary)

In [ ]:
# NET MARGIN BY CLUSTER (K-MEANS)
# Calculate net margin as (gross_spend - return_value) / gross_spend * 100

margin_analysis = df_final.groupby('cluster').agg(
    customer_count=('customer_id', 'count'),
    total_gross_spend=('gross_spend', 'sum'),
    total_return_value=('return_value', 'sum'),
    total_net_spend=('net_amount', 'sum'),
    avg_gross_spend=('gross_spend', 'mean'),
    avg_return_value=('return_value', 'mean'),
    avg_net_amount=('net_amount', 'mean'),
).reset_index()

# Calculate percentage margin
margin_analysis['net_margin_pct'] = (
    (margin_analysis['total_gross_spend'] - margin_analysis['total_return_value']) 
    / margin_analysis['total_gross_spend'] * 100
).round(2)

margin_analysis['avg_net_margin_pct'] = (
    (margin_analysis['avg_gross_spend'] - margin_analysis['avg_return_value']) 
    / margin_analysis['avg_gross_spend'] * 100
).round(2)

# Also calculate return rates
margin_analysis['return_rate_pct'] = (
    (margin_analysis['total_return_value'] / margin_analysis['total_gross_spend']) * 100
).round(2)

# Sort by descending net margin
margin_analysis = margin_analysis.sort_values('net_margin_pct', ascending=False)

print("=" * 100)
print("NET MARGIN BY CLUSTER (K-MEANS - 4 Clusters)")
print("=" * 100)
print("\nCLUSTER-LEVEL TOTALS:")
print(margin_analysis[['cluster', 'customer_count', 'total_gross_spend', 'total_return_value', 'net_margin_pct', 'return_rate_pct']].to_string(index=False))

print("\n" + "=" * 100)
print("CLUSTER AVERAGES (average values per customer):")
print("=" * 100)
print(margin_analysis[['cluster', 'customer_count', 'avg_gross_spend', 'avg_return_value', 'avg_net_margin_pct']].to_string(index=False))

print("\n" + "=" * 100)
print("INTERPRETATION:")
print("=" * 100)
for idx, row in margin_analysis.iterrows():
    cluster_id = int(row['cluster'])
    margin = row['net_margin_pct']
    return_rate = row['return_rate_pct']
    customers = int(row['customer_count'])
    
    if margin >= 70:
        profitability = "Very High ✓✓✓"
    elif margin >= 60:
        profitability = "High ✓✓"
    elif margin >= 50:
        profitability = "Medium ✓"
    else:
        profitability = "Low ✗"
    
    print(f"\nCluster {cluster_id}: {customers:,} customers")
    print(f"  - Net margin: {margin}% ({profitability})")
    print(f"  - Return rate: {return_rate}%")
    print(f"  - Average spend per customer: EUR {row['avg_gross_spend']:.2f}")

print("\n" + "=" * 100)

In [ ]:
# QUICK MARGIN PRINT
summary_table = margin_analysis[['cluster', 'customer_count', 'net_margin_pct', 'return_rate_pct']].copy()
summary_table.columns = ['Cluster', 'Customers', 'Net Margin %', 'Return Rate %']
print(summary_table.to_string(index=False))

# DBSCAN

In [ ]:
# Plot k-distance knee to find optimal epsilon (eps)
def plot_k_distance(data, k=30):
    neighbors = NearestNeighbors(n_neighbors=k)
    neighbors_fit = neighbors.fit(data)
    
    distances, indices = neighbors_fit.kneighbors(data)

    k_distances = np.sort(distances[:, k-1], axis=0)
    
    plt.figure(figsize=(10, 6))
    plt.plot(k_distances)
    plt.title(f'K-Distance Graph (k={k})', fontsize=14)
    plt.xlabel('Data Points (sorted by distance)', fontsize=12)
    plt.ylabel(f'{k}-th Nearest Neighbor Distance (Potential eps)', fontsize=12)
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.show()

plot_k_distance(X_scaled, k=30)

In [ ]:
# INITIALIZE DBSCAN
dbscan = DBSCAN(eps=1.8, min_samples=30)

clusters_dbscan = dbscan.fit_predict(X_scaled)

df_final['cluster_dbscan'] = clusters_dbscan

n_clusters_ = len(set(clusters_dbscan)) - (1 if -1 in clusters_dbscan else 0)
n_noise_ = list(clusters_dbscan).count(-1)

print(f'Estimated number of clusters: {n_clusters_}')
print(f'Estimated number of noise points: {n_noise_}')

dbscan_summary = df_final[df_final['cluster_dbscan'] != -1].groupby('cluster_dbscan')[num_cols].mean()
print("\n--- DBSCAN Cluster Averages ---")
print(dbscan_summary)

# OPTICS

In [ ]:
# OPTICS will find the best epsilon for each cluster automatically
optics_model = OPTICS(min_samples=15, xi=0.05, min_cluster_size=0.05)
df_final['cluster_optics'] = optics_model.fit_predict(X_scaled)

print("--- OPTICS Cluster Sizes ---")
print(df_final['cluster_optics'].value_counts())

### Hairball problem, DBSCAN and OPTICS won't work as the data is a "Global Density" mass -> no clear gaps between customer groups. Need centroid-based or distribution-based clustering

# Gaussian Mixture Model

In [ ]:
# 1. Initialize GMM
gmm = GaussianMixture(n_components=4, random_state=42)

df_final['cluster_gmm'] = gmm.fit_predict(X_scaled)

gmm_summary = df_final.groupby('cluster_gmm')[num_cols].mean().round(2)
print("--- GMM Cluster Averages ---")
print(gmm_summary)

In [ ]:
# cluster sizes
print("--- Raw Cluster Sizes ---")
print(df_final['cluster_gmm'].value_counts())

print("\n--- Percentage Breakdown ---")
print(df_final['cluster_gmm'].value_counts(normalize=True) * 100)

In [ ]:
# Radar chart for GMM clusters
# Uses only columns actually present in gmm_summary (dynamic filter),
# and generates colors automatically based on the number of clusters.

# Column → readable label map (add aliases here if the CSV uses different column names)
RADAR_COLS = {
    'age_years':                    'Age',
    'net_amount':                   'Net Amount',
    'net_spend_recalc':             'Net Amount',   # alias
    'monetary':                     'Net Amount',   # alias
    'transaction_count':            'Tx Count',
    'transaction_count_recalc':     'Tx Count',     # alias
    'frequency':                    'Tx Count',     # alias
    'return_rate_tx':               'Return Rate',
    'return_rate':                  'Return Rate',  # alias
    'nl_open_rate':                 'Email Open',
    'engagement_score':             'Email Engage', # alias
    'recency_days':                 'Recency',
    'avg_ticket':                   'Avg Ticket',
    'avg_basket_value':             'Avg Basket',   # alias
    'gross_spend':                  'Gross Spend',
}

# Select columns available in gmm_summary (skip alias duplicates)
seen_labels = set()
selected_cols   = []
selected_labels = []
for col, lbl in RADAR_COLS.items():
    if col in gmm_summary.columns and lbl not in seen_labels:
        selected_cols.append(col)
        selected_labels.append(lbl)
        seen_labels.add(lbl)

if len(selected_cols) < 3:
    print(f"Too few columns available for radar chart ({selected_cols}). Skipping.")
else:
    stats      = gmm_summary[selected_cols].copy()
    col_range  = stats.max() - stats.min()
    stats_norm = stats.copy()
    for c in selected_cols:
        if col_range[c] > 0:
            stats_norm[c] = (stats[c] - stats[c].min()) / col_range[c]
        else:
            stats_norm[c] = 0.5   # constant column → center at 0.5

    data_array = stats_norm.values
    num_vars   = len(selected_cols)
    angles     = np.linspace(0, 2 * np.pi, num_vars, endpoint=False).tolist()
    data_plot  = np.concatenate((data_array, data_array[:, [0]]), axis=1)
    angles_plot = angles + [angles[0]]

    # Dynamic palette based on number of clusters
    palette_base = ['#552583', '#63727b', '#fdb927', '#007a33',
                    '#E8883A', '#9B7EC8', '#D94F4F', '#2ECC71',
                    '#3498DB', '#E67E22']
    n_clusters = len(data_plot)
    colors = [palette_base[i % len(palette_base)] for i in range(n_clusters)]

    fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))

    for i in range(n_clusters):
        cid   = gmm_summary.index[i]
        label = f'Cluster {cid}'
        ax.plot(angles_plot, data_plot[i], color=colors[i], linewidth=2, label=label)
        ax.fill(angles_plot, data_plot[i], color=colors[i], alpha=0.15)

    ax.set_thetagrids(np.degrees(angles), selected_labels)
    ax.set_theta_offset(np.pi / 2)
    ax.set_theta_direction(-1)
    ax.set_ylim(0, 1)
    plt.title('GMM Cluster Personas (Normalized Comparison)', size=15, y=1.1)
    plt.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))
    plt.tight_layout()
    plt.show()

# Hierarchical clustering

In [ ]:
# VISUALIZE THE DENDROGRAM
plt.figure(figsize=(10, 7))
plt.title("Customer Dendrogram")
dendrogram = sch.dendrogram(sch.linkage(X_scaled, method='ward'))
plt.show()

# RUN THE CLUSTERING
hc = AgglomerativeClustering(n_clusters=4, metric='euclidean', linkage='ward')
df_final['cluster_hc'] = hc.fit_predict(X_scaled)

hc_summary = df_final.groupby('cluster_hc')[num_cols].mean().round(2)
print("--- Hierarchical Clustering Averages ---")
print(hc_summary)

print("\n--- Cluster Sizes ---")
print(df_final['cluster_hc'].value_counts().sort_index())

In [ ]:
# See which provinces have the most VIPs (Cluster 2)
if 'province' in df_final.columns:
    vip_provinces = df_final[df_final['cluster_hc'] == 2]['province'].value_counts().head(10)
    print("Top 10 Provinces for VIPs:\n", vip_provinces)
else:
    print("Skipping VIP province view: 'province' column not available in this dataset.")

In [ ]:
# Find provinces with a high number of VIPs relative to national average
if 'province' in df_final.columns:
    total_vips = len(df_final[df_final['cluster_hc'] == 2])
    total_customers = len(df_final)
    global_vip_rate = total_vips / total_customers

    province_stats = df_final.groupby('province').agg(
        total_in_province=('customer_id', 'count'),
        vips_in_province=('cluster_hc', lambda x: (x == 2).sum())
    )

    # Filter out provinces with very few customers to avoid statistical noise
    min_sample_size = 50
    province_stats = province_stats[province_stats['total_in_province'] >= min_sample_size].copy()

    province_stats['actual_vip_rate'] = province_stats['vips_in_province'] / province_stats['total_in_province']
    province_stats['vip_index'] = (((province_stats['actual_vip_rate'] / global_vip_rate) * 100)).round(1)

    top_hotspots = province_stats.sort_values('vip_index', ascending=False).head(10)

    print(f"Global VIP Baseline: {global_vip_rate:.2%}")
    print("\n--- Top 10 Over-Performing Provinces (Hot Spots) ---")
    print(top_hotspots[['total_in_province', 'vips_in_province', 'vip_index']])
else:
    print("Skipping province hotspot analysis: 'province' column not available in this dataset.")

# Evaluation of HC

In [ ]:
X = X_scaled  # The features used for clustering
labels = df_final['cluster_hc']

# 1. Calculate the Scores
sil = silhouette_score(X, labels)
ch = calinski_harabasz_score(X, labels)
db = davies_bouldin_score(X, labels)

print(f"--- Hierarchical Clustering Evaluation (k=4) ---")
print(f"Silhouette Score: {sil:.4f}  (Higher is better, >0.2 is good)")
print(f"Calinski-Harabasz: {ch:.2f} (Higher is better)")
print(f"Davies-Bouldin:   {db:.4f}  (Lower is better, closer to 0 is better)")

# 2. Comparative Analysis 
eval_results = []
for k in [3, 4, 5, 6]:
    model = AgglomerativeClustering(n_clusters=k)
    temp_labels = model.fit_predict(X)
    
    eval_results.append({
        'k': k,
        'Silhouette': silhouette_score(X, temp_labels),
        'Calinski-Harabasz': calinski_harabasz_score(X, temp_labels),
        'Davies-Bouldin': davies_bouldin_score(X, temp_labels)
    })

eval_df = pd.DataFrame(eval_results)
print("\n--- Comparative Metrics by Number of Clusters ---")
print(eval_df.to_string(index=False))